In [2]:
import bb_gym
import numpy as np
import gymnasium as gym 
# Import one of the canonical ranges of physical error rate from utils

from bb_gym.utils import GEOMETRIC5
l = 6
m = 6
k = 6

#env = GymEnv("bb_gym/bbcode-ldpc-v0", 
env = gym.make('bb_gym/bbcode-ldpc-v0',
                          l = l, 
                          m = m, 
                          errorRange = GEOMETRIC5, 
                          minimumNumberOfLogicalQubits = k, # Use 6 for the 6,6 code, 10 for the 9,6 and 15,3 codes
                          numberOfIterations = 20, # How many iterations of BP to run
                          numberOfSamples = 10, # How many Pauli errors to sample in each physical error point
                          rewardEngineering = True, # This determines whether or not we divide the integral by the width of the region of interest
                          bitFlipping = True, # If set to True, each action is interpreted as flipping a single bit. If False, we completely ignore the present state and rewrite the polynomials. Good for debugging or for a stateless environment.
                          resetType = "zero", # For this example, for reset to the all zero code. Replace with "random3" for actual RL, otherwise the agent only learns from the all 0 state.
                          stepsLimit = 12) # This determines the number of steps that the environment can progress, afterwhich it resets itself. So this should be in the order of max(l,m). In the paper we use 30 as one size fits all for all the code sizes.

# Check that the code we get on reset is the all 0 code:
print(env.reset()[0]['code'])
print(np.all(env.reset()[0]['code'] == 0))
# The action space should be l + 1 + l + 1 + m + 1 + m + 1 (one no-op bit per polynomial)
print(env.action_space.shape)

[0. 0. 0. ... 0. 0. 0.]
True
(28,)


In [3]:
# So an action looks like a concatenation of one-hot binary vectors
# Let's build the 6,6,6 code in two actions:
env.reset()
aX = np.zeros(l + 1)
aY = np.zeros(m + 1)
bX = np.zeros(l + 1)
bY = np.zeros(m + 1)
aX[3] = 1
aY[1] = 1
bX[1] = 1  
bY[3] = 1
action = np.concatenate([aX, bX, aY, bY])
print(f"An action is a flat concatenation of the per-polynomial actions: {action}")
observation, reward, terminated, truncated, info = env.step(action = action)
print(f"Reward after the first action: {reward}.")
print(f"The number of logical qubits for this code was calculated as: {observation['k'][0]}, and the minimum number of logical qubits we initialized the environment is: {k}.")
if observation['k'][0] < k:
    print(f"So the reward will be calculated as {observation['k'][0]} - {k} / {k} == {(observation['k'][0] - k) / k}. Check: {(reward - ((observation['k'][0] - k) / k)) < np.finfo(np.float32).eps}")
else:
    print(f"If the number of logical qubits exceeds the minimum, the environment will go into code-decoder evaluation, and the reward will be non negative.")


# An action should have only one non zero bit for each polynomial, so let's first zero them out:
aX = 0 * aX
aY = 0 * aY
bX = 0 * bX
bY = 0 * bY

#No op for aX
aX[l] = 1 
aY[2] = 1
bX[2] = 1
# No op for bY
bY[m] = 1
action = np.concatenate([aX, bX, aY, bY])
observation, reward, terminated, truncated, info = env.step(action = action)
print(f"Reward after the second action: {reward}")

An action is a flat concatenation of the per-polynomial actions: [0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 0. 0. 0.]
Reward after the first action: -0.3333333333333333.
The number of logical qubits for this code was calculated as: 4.0, and the minimum number of logical qubits we initialized the environment is: 6.
So the reward will be calculated as 4.0 - 6 / 6 == -0.3333333432674408. Check: True
Reward after the second action: 0.36755922579561745


In [ ]:
from torch.distributions import OneHotCategorical
import torch

# How would this look like in a pytorch setting ?

# The last layer of a model (to be learned) will be used as logits for a one hot categorical distribution, which will be used in exploration mode
logitsForXPolynomials = torch.tensor(([0.3] * (l + 1)))
logitsForYPolynomials = torch.tensor(([0.3] * (m + 1))) #Just in case we want to use l != m
aXDistribution = OneHotCategorical(logits = logitsForXPolynomials)
aYDistribution = OneHotCategorical(logits = logitsForYPolynomials)

# Each distribution is sampled, the results are concatenated and used as an action:
action = np.concatenate([aXDistribution.sample().numpy(), aYDistribution.sample().numpy(), aXDistribution.sample().numpy(), aYDistribution.sample().numpy()]) # Obviously in real life we want these distributions to be independent, but for a demo just the dimensions matter
print(f"Sanity check, the action needs to be a flat array of length {2*l+2*m+4}, and it is of length: {len(action)}.")
observation, reward, terminated, truncated, info = env.step(action = action)
print(f"Reward after a random action: {reward}")



Sanity check, the action needs to be a flat array of length 28, and it is of length: 28.
Reward after a random action: 0.231195589431981
